# Paper Analysis Notebook

This notebook loads evaluation artifacts from RecBole and LLM evaluations,
normalizes them into the shared ranking schema, and generates paper-ready
tables and figures under `data/recbole/paper_results/`.

In [ ]:
import sys
from pathlib import Path

# Allow imports from src regardless of CWD
_cwd = Path.cwd()
if (_cwd / "src").exists():
    sys.path.insert(0, str(_cwd / "src"))
elif (_cwd.parent / "src").exists():
    sys.path.insert(0, str(_cwd.parent / "src"))

import json  # noqa: E402
import logging  # noqa: E402
import warnings  # noqa: E402

import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import seaborn as sns  # noqa: E402

# Shared helpers
from stability.evaluation import (  # noqa: E402
    average_popularity_at_k,
    bootstrap_ci,
    build_title_to_item_id_mapping,
    compute_per_user_accuracy,
    compute_tail_items,
    genre_coverage_at_k,
    genre_entropy_at_k,
    gini_index_at_k,
    item_coverage_at_k,
    load_catalog,
    load_genre_mapping,
    load_popularity,
    normalize_llm_results,
    ranking_rows_to_dataframe,
    shannon_entropy_at_k,
    tail_percentage_at_k,
)
from stability.preprocessing import normalize_movie_title  # noqa: E402

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO)
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("colorblind")

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
DATA_PATH = Path("../data")
RECBOLE_PATH = DATA_PATH / "recbole"
OUTPUT_PATH = DATA_PATH / "output"
PAPER_RESULTS_PATH = RECBOLE_PATH / "paper_results"
PAPER_RESULTS_PATH.mkdir(parents=True, exist_ok=True)

print(f"Paper results will be written to: {PAPER_RESULTS_PATH}")

In [ ]:
item_path = RECBOLE_PATH / "redial" / "redial.item"
train_inter_path = RECBOLE_PATH / "redial" / "redial.train.inter"
metadata_path = DATA_PATH / "processed" / "movies_metadata_tmdb.csv"

id_to_title_catalog, catalog_item_ids = load_catalog(item_path)
popularity_counts = load_popularity(train_inter_path, catalog_item_ids)
tail_item_ids = compute_tail_items(popularity_counts, tail_fraction=0.1)
genre_mapping = load_genre_mapping(metadata_path, id_to_title_catalog)
title_to_item_id = build_title_to_item_id_mapping(item_path)

print(f"Catalog: {len(catalog_item_ids)} items")
print(f"Tail: {len(tail_item_ids)} items")
print(f"Genres mapped: {sum(1 for g in genre_mapping.values() if g)}")

In [ ]:
def discover_llm_result_files(output_path: Path) -> list[Path]:
    """Return complete LLM evaluation parquet files, newest last for deduping."""
    files = set(output_path.glob("evaluation_results_*.parquet"))
    files.update(output_path.glob("evaluation_results_named_pools_*.parquet"))
    filtered = []
    for path in files:
        name = path.name
        if name.startswith("evaluation_results_recbole_"):
            continue
        if "partial" in name:
            continue
        filtered.append(path)
    return sorted(filtered, key=lambda p: (p.stat().st_mtime, p.name))


LLM_PARQUET_FILES = discover_llm_result_files(OUTPUT_PATH)
print("LLM result files:")
for p in LLM_PARQUET_FILES:
    print(f"  - {p.name}")

llm_dfs = []
for p in LLM_PARQUET_FILES:
    df = pd.read_parquet(p)
    print(f"Loaded {p.name}: {len(df)} rows")
    normalized = normalize_llm_results(df, title_to_item_id=title_to_item_id)
    normalized["_source_file"] = p.name
    normalized["_source_mtime"] = p.stat().st_mtime
    llm_dfs.append(normalized)

if llm_dfs:
    llm_normalized = pd.concat(llm_dfs, ignore_index=True)
    print(f"Combined normalized LLM rows: {len(llm_normalized)}")
    before = len(llm_normalized)
    llm_normalized = (
        llm_normalized.sort_values(["_source_mtime", "_source_file"])
        .drop_duplicates(
            subset=[
                "model",
                "retriever_type",
                "retriever_model",
                "n_candidates",
                "prompt_idx",
            ],
            keep="last",
        )
        .drop(columns=["_source_file", "_source_mtime"])
    )
    if len(llm_normalized) < before:
        print(f"Deduplicated LLM rows: {before} -> {len(llm_normalized)}")
    print(f"Normalized LLM rows: {len(llm_normalized)}")
else:
    llm_normalized = pd.DataFrame()
    print("WARNING: No LLM parquet files found")

In [ ]:
RECBOLE_RANKED_FILES = {
    "cf": RECBOLE_PATH / "evaluation_results" / "cf_ranked_lists.parquet",
    "sequential": RECBOLE_PATH / "evaluation_results" / "seq_ranked_lists.parquet",
}

recbole_ranked_frames = []
for model_family, p in RECBOLE_RANKED_FILES.items():
    if not p.exists():
        print(f"WARNING: {p.name} not found - will use legacy fallback if available")
        continue
    df = pd.read_parquet(p)
    df = ranking_rows_to_dataframe(df.to_dict("records"))
    print(f"Loaded {p.name}: {len(df)} rows")
    recbole_ranked_frames.append(df)

if recbole_ranked_frames:
    df_recbole_ranked = pd.concat(recbole_ranked_frames, ignore_index=True)
    print(f"RecBole ranked artifact rows: {len(df_recbole_ranked)}")
else:
    RECBOLE_RERANKING_FILES = {
        "cf": OUTPUT_PATH / "evaluation_results_recbole_cf.parquet",
        "sequential": OUTPUT_PATH / "evaluation_results_recbole_sequential.parquet",
    }
    recbole_reranking_rows = []

    for model_family, p in RECBOLE_RERANKING_FILES.items():
        if not p.exists():
            print(f"WARNING: {p.name} not found - skipping")
            continue
        df = pd.read_parquet(p)
        print(f"Loaded {p.name}: {len(df)} rows")
        for _, row in df.iterrows():
            pred_titles = row.get("pred_items", [])
            gt_titles = row.get("ground_truth", [])
            ranked_item_ids = []
            ground_truth_item_ids = []
            for title in pred_titles:
                norm_title = normalize_movie_title(str(title))
                item_id = (
                    title_to_item_id.get(norm_title) if norm_title is not None else None
                )
                if item_id is not None:
                    ranked_item_ids.append(item_id)
            for title in gt_titles:
                norm_title = normalize_movie_title(str(title))
                item_id = (
                    title_to_item_id.get(norm_title) if norm_title is not None else None
                )
                if item_id is not None:
                    ground_truth_item_ids.append(item_id)
            n_candidates_raw = row.get("n_candidates")
            n_candidates = (
                int(n_candidates_raw[1:])
                if isinstance(n_candidates_raw, str)
                and n_candidates_raw.startswith("c")
                else n_candidates_raw
            )
            recbole_reranking_rows.append(
                {
                    "model_type": model_family,
                    "model": row.get("model"),
                    "eval_method": "cbf_reranking",
                    "retriever_type": "cbf",
                    "retriever_model": "CBF",
                    "reranker_type": model_family,
                    "reranker_model": row.get("model"),
                    "n_candidates": n_candidates,
                    "user_id": row.get("user_id"),
                    "prompt_idx": row.get("prompt_idx"),
                    "ranked_item_ids": ranked_item_ids,
                    "ranked_titles": list(pred_titles),
                    "ranked_scores": [],
                    "ground_truth_item_ids": ground_truth_item_ids,
                    "ground_truth_titles": list(gt_titles),
                }
            )

    if recbole_reranking_rows:
        df_recbole_ranked = ranking_rows_to_dataframe(recbole_reranking_rows)
        print(f"Legacy RecBole CBF+reranking rows: {len(df_recbole_ranked)}")
    else:
        df_recbole_ranked = pd.DataFrame()

In [ ]:
ACCURACY_METRICS = {"hit_rate", "mrr", "precision", "recall", "f1", "ndcg"}
standalone_aggregate_rows = []

for parquet_path in [
    RECBOLE_PATH / "evaluation_results" / "cf_aggregate_metrics.parquet",
    RECBOLE_PATH / "evaluation_results" / "seq_aggregate_metrics.parquet",
]:
    if not parquet_path.exists():
        continue
    df_parquet = pd.read_parquet(parquet_path)
    df_parquet = df_parquet[
        (df_parquet["eval_method"] == "standalone")
        & (df_parquet["metric"].isin(ACCURACY_METRICS))
    ]
    standalone_aggregate_rows.extend(df_parquet.to_dict("records"))
    print(
        f"Loaded standalone aggregates from {parquet_path.name}: {len(df_parquet)} rows"
    )

if not standalone_aggregate_rows:
    for json_path, family in [
        (RECBOLE_PATH / "evaluation_results" / "recbole_final_results.json", "cf"),
        (
            RECBOLE_PATH / "evaluation_results" / "sequential_final_results.json",
            "sequential",
        ),
    ]:
        if json_path.exists():
            with json_path.open() as f:
                data = json.load(f)
            for model_name, result in data.items():
                test_result = result.get("test_result", {})
                for metric_key, value in test_result.items():
                    if "@" not in metric_key:
                        continue
                    metric_name, k_str = metric_key.rsplit("@", 1)
                    try:
                        k = int(k_str)
                    except ValueError:
                        continue
                    metric_name = metric_name.replace("hit", "hit_rate")
                    if metric_name not in ACCURACY_METRICS:
                        continue
                    standalone_aggregate_rows.append(
                        {
                            "model_type": family,
                            "model": model_name,
                            "eval_method": "standalone",
                            "retriever_type": family,
                            "retriever_model": model_name,
                            "reranker_type": "none",
                            "reranker_model": "none",
                            "n_candidates": len(catalog_item_ids),
                            "metric": metric_name,
                            "k": k,
                            "value": float(value),
                        }
                    )
            print(f"Loaded standalone aggregates from {json_path.name}")
        else:
            print(f"WARNING: {json_path.name} not found")

unified_csv = RECBOLE_PATH / "evaluation_results" / "unified_comparison.csv"
if unified_csv.exists() and not standalone_aggregate_rows:
    df_unified = pd.read_csv(unified_csv)
    df_unified = df_unified[df_unified["eval_method"] == "standalone"]
    for _, row in df_unified.iterrows():
        for metric_name in ["hit_rate", "mrr", "precision", "recall", "ndcg"]:
            if metric_name in row:
                standalone_aggregate_rows.append(
                    {
                        "model_type": row["model_type"],
                        "model": row["model"],
                        "eval_method": "standalone",
                        "retriever_type": row["model_type"].lower()
                        if isinstance(row["model_type"], str)
                        else row["model_type"],
                        "retriever_model": row["model"],
                        "reranker_type": "none",
                        "reranker_model": "none",
                        "n_candidates": row["n_candidates"],
                        "metric": metric_name,
                        "k": row["k"],
                        "value": float(row[metric_name]),
                    }
                )
    print(f"Loaded standalone aggregates from {unified_csv.name}")

df_standalone_agg = pd.DataFrame(standalone_aggregate_rows)
print(f"Standalone aggregate rows: {len(df_standalone_agg)}")

In [ ]:
METRIC_CUTOFFS = [1, 5, 10]

# --- LLM per-user accuracy ---
if not llm_normalized.empty:
    llm_acc = compute_per_user_accuracy(llm_normalized, k_values=METRIC_CUTOFFS)
    print(f"LLM per-user accuracy computed: {len(llm_acc)} rows")
else:
    llm_acc = pd.DataFrame()

# --- RecBole per-user accuracy from ranked artifacts or legacy fallback ---
if not df_recbole_ranked.empty:
    reb_acc = compute_per_user_accuracy(df_recbole_ranked, k_values=METRIC_CUTOFFS)
    print(f"RecBole per-user accuracy computed: {len(reb_acc)} rows")
else:
    reb_acc = pd.DataFrame()

per_user_frames = []
if not llm_acc.empty:
    per_user_frames.append(llm_acc)
if not reb_acc.empty:
    per_user_frames.append(reb_acc)

if per_user_frames:
    df_per_user = pd.concat(per_user_frames, ignore_index=True)
    print(f"Combined per-user rows: {len(df_per_user)}")
else:
    df_per_user = pd.DataFrame()
    print("WARNING: No per-user data available")

In [ ]:
def compute_beyond_accuracy_for_group(
    df_group: pd.DataFrame, k_values: list[int]
) -> list[dict]:
    """Return beyond-accuracy aggregate rows for a DataFrame group."""
    ranked_item_ids_list = df_group["ranked_item_ids"].tolist()
    rows = []
    for k in k_values:
        rows.append(
            {
                "metric": "item_coverage",
                "k": k,
                "value": item_coverage_at_k(ranked_item_ids_list, catalog_item_ids, k),
            }
        )
        rows.append(
            {
                "metric": "average_popularity",
                "k": k,
                "value": average_popularity_at_k(
                    ranked_item_ids_list, popularity_counts, k
                ),
            }
        )
        rows.append(
            {
                "metric": "gini_index",
                "k": k,
                "value": gini_index_at_k(ranked_item_ids_list, catalog_item_ids, k),
            }
        )
        rows.append(
            {
                "metric": "shannon_entropy",
                "k": k,
                "value": shannon_entropy_at_k(
                    ranked_item_ids_list, catalog_item_ids, k
                ),
            }
        )
        rows.append(
            {
                "metric": "tail_percentage",
                "k": k,
                "value": tail_percentage_at_k(ranked_item_ids_list, tail_item_ids, k),
            }
        )
        rows.append(
            {
                "metric": "genre_coverage",
                "k": k,
                "value": genre_coverage_at_k(ranked_item_ids_list, genre_mapping, k),
            }
        )
        rows.append(
            {
                "metric": "genre_entropy",
                "k": k,
                "value": genre_entropy_at_k(ranked_item_ids_list, genre_mapping, k),
            }
        )
    return rows


ba_rows = []
if not df_per_user.empty:
    group_keys = [
        "model_type",
        "model",
        "eval_method",
        "retriever_type",
        "retriever_model",
        "reranker_type",
        "reranker_model",
        "n_candidates",
    ]
    for keys, group in df_per_user.groupby(group_keys, dropna=False):
        meta = dict(zip(group_keys, keys, strict=False))  # type: ignore[arg-type]
        for r in compute_beyond_accuracy_for_group(group, METRIC_CUTOFFS):
            ba_rows.append({**meta, **r})

df_ba = pd.DataFrame(ba_rows)
print(f"Beyond-accuracy aggregate rows: {len(df_ba)}")

In [ ]:
def bootstrap_accuracy_ci(
    df_group: pd.DataFrame,
    metric_name: str,
    k: int,
    n_bootstraps: int = 1000,
    seed: int = 42,
) -> dict:
    """Bootstrap CI for a single accuracy metric over users."""
    col = f"{metric_name}@{k}"
    if col not in df_group.columns or df_group.empty:
        return {
            "metric": metric_name,
            "k": k,
            "lower": np.nan,
            "upper": np.nan,
            "std": np.nan,
        }
    sub = df_group[[col]].dropna()
    if sub.empty:
        return {
            "metric": metric_name,
            "k": k,
            "lower": np.nan,
            "upper": np.nan,
            "std": np.nan,
        }
    result = bootstrap_ci(
        sub,
        metric_fn=lambda df: float(df[col].mean()),
        n_bootstraps=n_bootstraps,
        seed=seed,
    )
    return {
        "metric": metric_name,
        "k": k,
        "lower": result["lower"],
        "upper": result["upper"],
        "std": result["std"],
    }


bootstrap_rows = []
if not df_per_user.empty:
    group_keys = [
        "model_type",
        "model",
        "eval_method",
        "retriever_type",
        "retriever_model",
        "reranker_type",
        "reranker_model",
        "n_candidates",
    ]
    for keys, group in df_per_user.groupby(group_keys, dropna=False):
        meta = dict(zip(group_keys, keys, strict=False))  # type: ignore[arg-type]
        for metric_name in ["ndcg", "hit_rate", "mrr", "precision", "recall"]:
            ci = bootstrap_accuracy_ci(
                group, metric_name, 10, n_bootstraps=1000, seed=42
            )
            bootstrap_rows.append({**meta, **ci})

df_bootstrap = pd.DataFrame(bootstrap_rows)
print(f"Bootstrap CI rows: {len(df_bootstrap)}")

In [ ]:
agg_accuracy_rows = []

# From per-user data (LLM + RecBole ranked artifacts / fallback)
if not df_per_user.empty:
    group_keys = [
        "model_type",
        "model",
        "eval_method",
        "retriever_type",
        "retriever_model",
        "reranker_type",
        "reranker_model",
        "n_candidates",
    ]
    for keys, group in df_per_user.groupby(group_keys, dropna=False):
        meta = dict(zip(group_keys, keys, strict=False))  # type: ignore[arg-type]
        for k in METRIC_CUTOFFS:
            for metric_name in ["hit_rate", "mrr", "precision", "recall", "f1", "ndcg"]:
                col = f"{metric_name}@{k}"
                value = float(group[col].mean()) if col in group.columns else np.nan
                agg_accuracy_rows.append(
                    {
                        **meta,
                        "metric": metric_name,
                        "k": k,
                        "value": value,
                        "_source": "per_user",
                    }
                )

# From standalone aggregate data
if not df_standalone_agg.empty:
    for _, row in df_standalone_agg.iterrows():
        record = row.to_dict()
        record["_source"] = "standalone"
        agg_accuracy_rows.append(record)

df_agg_accuracy = pd.DataFrame(agg_accuracy_rows)
print(f"Aggregate accuracy rows: {len(df_agg_accuracy)}")

if not df_agg_accuracy.empty:
    df_agg_accuracy["_source_priority"] = (
        df_agg_accuracy["_source"].map({"per_user": 0, "standalone": 1}).fillna(2)
    )
    df_agg_accuracy = (
        df_agg_accuracy.sort_values("_source_priority")
        .drop_duplicates(
            subset=[
                c
                for c in df_agg_accuracy.columns
                if c not in {"_source", "_source_priority", "value"}
            ],
            keep="first",
        )
        .drop(columns=["_source", "_source_priority"])
    )
    print(f"After dedup: {len(df_agg_accuracy)}")

In [ ]:
all_agg = []
if not df_agg_accuracy.empty:
    all_agg.append(df_agg_accuracy)
if not df_ba.empty:
    all_agg.append(df_ba)

df_all_agg = pd.concat(all_agg, ignore_index=True) if all_agg else pd.DataFrame()

print(f"Total aggregate metric rows: {len(df_all_agg)}")
print("Methods represented:")
if not df_all_agg.empty:
    print(
        df_all_agg[["model_type", "model", "eval_method", "n_candidates"]]
        .drop_duplicates()
        .head(20)
    )

In [ ]:
# Filter to CBF reranking at 250 candidates
mask_main = (
    (df_all_agg["eval_method"].isin(["cbf_reranking", "reranking"]))
    & (df_all_agg["retriever_type"] == "cbf")
    & (df_all_agg["retriever_model"] == "CBF")
    & (df_all_agg["n_candidates"] == 250)
    & (df_all_agg["k"] == 10)
)
df_main = df_all_agg[mask_main].copy()

if not df_main.empty:
    pivot_main = df_main.pivot_table(
        index=[
            "model_type",
            "model",
            "eval_method",
            "retriever_type",
            "retriever_model",
            "reranker_type",
            "reranker_model",
            "n_candidates",
        ],
        columns="metric",
        values="value",
        aggfunc="first",
    ).reset_index()
    if not df_bootstrap.empty:
        boot_ndcg = df_bootstrap[
            (df_bootstrap["k"] == 10) & (df_bootstrap["metric"] == "ndcg")
        ][
            [
                "model_type",
                "model",
                "eval_method",
                "retriever_type",
                "retriever_model",
                "reranker_type",
                "reranker_model",
                "n_candidates",
                "lower",
                "upper",
                "std",
            ]
        ]
        pivot_main = pivot_main.merge(
            boot_ndcg,
            on=[
                "model_type",
                "model",
                "eval_method",
                "retriever_type",
                "retriever_model",
                "reranker_type",
                "reranker_model",
                "n_candidates",
            ],
            how="left",
        )
    print(f"Main CBF+reranking table rows: {len(pivot_main)}")
    print(
        pivot_main[
            ["model_type", "model", "ndcg", "hit_rate", "mrr", "precision", "recall"]
        ].head(20)
    )
    pivot_main.to_csv(PAPER_RESULTS_PATH / "table_cbf_reranking_250.csv", index=False)
    try:
        pivot_main.to_latex(
            PAPER_RESULTS_PATH / "table_cbf_reranking_250.tex",
            index=False,
            float_format="%.4f",
        )
    except Exception as e:
        print(f"LaTeX export note: {e}")
else:
    print("WARNING: No data for main CBF+reranking table")

In [ ]:
# Focus on LLM rerankers across all retrievers at 250 candidates.
# RecBole rerankers on CBF already appear in the main CBF+reranking table.
mask_ret = (
    (df_all_agg["eval_method"].isin(["reranking"]))
    & (df_all_agg["n_candidates"] == 250)
    & (df_all_agg["k"] == 10)
    & (df_all_agg["reranker_type"] == "llm")
)
df_ret = df_all_agg[mask_ret].copy()

if not df_ret.empty:
    pivot_ret = df_ret.pivot_table(
        index=[
            "model_type",
            "model",
            "retriever_type",
            "retriever_model",
            "reranker_type",
            "reranker_model",
            "n_candidates",
        ],
        columns="metric",
        values="value",
        aggfunc="first",
    ).reset_index()
    print(f"Retrieval x reranker table rows: {len(pivot_ret)}")
    print(
        pivot_ret[
            [
                "model_type",
                "model",
                "retriever_type",
                "retriever_model",
                "reranker_model",
                "ndcg",
                "hit_rate",
                "mrr",
            ]
        ].head(20)
    )
    pivot_ret.to_csv(
        PAPER_RESULTS_PATH / "table_retrieval_x_reranker_250.csv", index=False
    )
    try:
        pivot_ret.to_latex(
            PAPER_RESULTS_PATH / "table_retrieval_x_reranker_250.tex",
            index=False,
            float_format="%.4f",
        )
    except Exception as e:
        print(f"LaTeX export note: {e}")
else:
    print("WARNING: No data for retrieval x reranker table")

In [ ]:
mask_standalone = (df_all_agg["eval_method"] == "standalone") & (df_all_agg["k"] == 10)
df_standalone = df_all_agg[mask_standalone].copy()

if not df_standalone.empty:
    pivot_standalone = df_standalone.pivot_table(
        index=["model_type", "model"],
        columns="metric",
        values="value",
        aggfunc="first",
    ).reset_index()
    print(f"Standalone baseline table rows: {len(pivot_standalone)}")
    print(
        pivot_standalone[
            ["model_type", "model", "ndcg", "hit_rate", "mrr", "precision", "recall"]
        ].head(20)
    )
    pivot_standalone.to_csv(
        PAPER_RESULTS_PATH / "table_standalone_baseline.csv", index=False
    )
    try:
        pivot_standalone.to_latex(
            PAPER_RESULTS_PATH / "table_standalone_baseline.tex",
            index=False,
            float_format="%.4f",
        )
    except Exception as e:
        print(f"LaTeX export note: {e}")
else:
    print("WARNING: No standalone baseline data")

In [ ]:
mask_ba = df_ba["k"] == 10
df_ba10 = df_ba[mask_ba].copy()

if not df_ba10.empty:
    pivot_ba = df_ba10.pivot_table(
        index=[
            "model_type",
            "model",
            "eval_method",
            "retriever_type",
            "retriever_model",
            "reranker_type",
            "reranker_model",
            "n_candidates",
        ],
        columns="metric",
        values="value",
        aggfunc="first",
    ).reset_index()
    print(f"Beyond-accuracy @10 table rows: {len(pivot_ba)}")
    print(
        pivot_ba[
            [
                "model_type",
                "model",
                "eval_method",
                "n_candidates",
                "item_coverage",
                "gini_index",
                "shannon_entropy",
                "tail_percentage",
            ]
        ].head(20)
    )
    pivot_ba.to_csv(PAPER_RESULTS_PATH / "table_beyond_accuracy_10.csv", index=False)
    try:
        pivot_ba.to_latex(
            PAPER_RESULTS_PATH / "table_beyond_accuracy_10.tex",
            index=False,
            float_format="%.4f",
        )
    except Exception as e:
        print(f"LaTeX export note: {e}")
else:
    print("WARNING: No beyond-accuracy data")

In [ ]:
# Plot NDCG@10 vs candidate size for CBF reranking methods that have multiple sizes
mask_sens = (
    (df_all_agg["eval_method"].isin(["cbf_reranking", "reranking"]))
    & (df_all_agg["metric"] == "ndcg")
    & (df_all_agg["k"] == 10)
    & (df_all_agg["n_candidates"].isin([0, 250, 500, 1000]))
)
df_sens = df_all_agg[mask_sens].copy()

if not df_sens.empty:
    fig, ax = plt.subplots(figsize=(8, 5))
    # Group by reranker model
    for rmodel, _grp in df_sens.groupby("reranker_model"):
        _grp = _grp.sort_values("n_candidates")
        ax.plot(_grp["n_candidates"], _grp["value"], marker="o", label=rmodel)
    ax.set_xlabel("Number of Candidates")
    ax.set_ylabel("NDCG@10")
    ax.set_title("Candidate-Size Sensitivity (CBF Retrieval + Reranker)")
    ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    ax.grid(visible=True)
    plt.tight_layout()
    fig.savefig(PAPER_RESULTS_PATH / "fig_candidate_sensitivity_ndcg10.png", dpi=300)
    fig.savefig(PAPER_RESULTS_PATH / "fig_candidate_sensitivity_ndcg10.pdf")
    plt.show()
    print("Saved candidate-size sensitivity figure")
else:
    print("WARNING: No candidate-size sensitivity data")

In [ ]:
summary_lines = [
    "# Paper Results Summary\n",
    f"Generated: {pd.Timestamp.now()}\n",
    "\n",
    "## Output Files\n",
]
for f in sorted(PAPER_RESULTS_PATH.iterdir()):
    summary_lines.append(f"- {f.name}\n")

summary_path = PAPER_RESULTS_PATH / "paper_summary.md"
with summary_path.open("w") as f:
    f.writelines(summary_lines)
print(f"Saved summary to {summary_path}")